# MCNN-QCC Datasets

Research on the application of multi-classification neural network in quantum correlation classification (MCNN-QCC)

<!-- % Using $ ... $ -->
<!-- % Using $$ ... $$ -->


In [1]:
import numpy as np


# 生成一个给定维度的子状态矩阵
def generate_sub_state(sub_dim):
    # 生成一个服从正态分布的随机矩阵
    real_part = np.random.normal(0, 1, [sub_dim, sub_dim])
    imag_part = np.random.normal(0, 1, [sub_dim, sub_dim])
    complex_matrix = (
        real_part + 1j * imag_part
    )  # 通过将实部和虚部组合，创建一个复数矩阵

    # 生成 Hermitian 矩阵
    hermitian_matrix = (complex_matrix + complex_matrix.T.conj()) / 2

    # 生成正定矩阵
    positive_definite_matrix = np.dot(hermitian_matrix, hermitian_matrix.T.conj())

    # 归一化为密度矩阵
    density_matrix = positive_definite_matrix / np.trace(positive_definite_matrix)

    return density_matrix

In [2]:
import numpy as np


# 生成随机复数态矢量
def random_complex_vector(size):

    # 生成实部和虚部都服从正态分布的随机列向量
    real_part = np.random.normal(0, 1, size)
    imag_part = np.random.normal(0, 1, size)

    # 组合实部和虚部，形成复数列向量
    state_vector = real_part + 1j * imag_part

    # 归一化列向量
    # norm = np.sqrt(np.sum(np.abs(state_vector)**2))
    norm = np.linalg.norm(state_vector)
    state_vector = state_vector / norm

    return state_vector


# 生成随机复数密度矩阵
def random_density_matrix(dim):

    state_vector = random_complex_vector(dim)

    # 生成密度矩阵：列向量与其共轭转置相乘
    density_matrix = np.outer(state_vector, np.conjugate(state_vector))

    return density_matrix

In [3]:
import numpy as np


# 验证给定的矩阵是否为半正定且迹为1
def verify_density_matrix(density_matrix):
    # 计算密度矩阵的迹
    trace = np.trace(density_matrix)

    # 计算密度矩阵的特征值
    eigenvalues = np.linalg.eigvals(density_matrix)

    # 验证所有特征值是否非负
    is_positive_semidefinite = np.all(eigenvalues >= -1e-10)  # 允许微小的数值误差

    # 验证迹是否为1
    is_trace_one = np.isclose(trace, 1)

    return is_positive_semidefinite, is_trace_one, eigenvalues, trace

# 3-qubit


## 全可分

$$
\rho=\sum_i p_i \rho_i^A \otimes \rho_i^B \otimes \rho_i^C
$$


In [4]:
import numpy as np


def generate_3_qubit_fully_separable_state(sub_dim=2, space_number=3, num_states=10):
    total_dim = sub_dim**space_number
    total_state = np.zeros((total_dim, total_dim), dtype=complex)

    # 生成随机权重并归一化
    weights = np.random.rand(num_states)
    weights /= np.sum(weights)

    for i in range(num_states):
        # 分别生成三个子系统的随机密度矩阵
        rho_A = generate_sub_state(sub_dim)
        rho_B = generate_sub_state(sub_dim)
        rho_C = generate_sub_state(sub_dim)

        # 计算张量积并累加到总态
        total_state += weights[i] * np.kron(np.kron(rho_A, rho_B), rho_C)

    # 归一化 total_state 的 trace 为 1
    total_state /= np.trace(total_state)

    return total_state

In [5]:
# 示例: 生成一个 3-qubit 全可分态密度矩阵
three_qubit_fully_separable_state = generate_3_qubit_fully_separable_state()

# 打印生成的密度矩阵形状
print(
    "Shape of the generated 4-qubit fully-separable state density matrix:",
    three_qubit_fully_separable_state.shape,
)

# 验证生成的密度矩阵
is_positive_semidefinite, is_trace_one, eigenvalues, trace = verify_density_matrix(
    three_qubit_fully_separable_state
)

# print("Generated 3-qubit fully-separable state density matrix:")
# print(four_qubit_fully_separable_state)
print("\nVerification Results:")
print("Is positive semidefinite:", is_positive_semidefinite)
print("Is trace one:", is_trace_one)
print("Eigenvalues:", eigenvalues)
print("Trace:", trace)

Shape of the generated 4-qubit fully-separable state density matrix: (8, 8)

Verification Results:
Is positive semidefinite: True
Is trace one: True
Eigenvalues: [0.2555554 +6.20483104e-18j 0.17499908+1.51244496e-18j
 0.14268925-7.42946752e-18j 0.05309757-8.54941734e-18j
 0.067429  -9.39152230e-18j 0.11628739-2.58856865e-18j
 0.09874322-2.27233644e-18j 0.0911991 +2.01967135e-18j]
Trace: (1-3.851859888774472e-34j)


## 二可分

$$
\rho= \sum_i p_i \rho_i^{A B} \otimes \rho_i^C + \sum_j p_j \rho_j^{A C} \otimes \rho_j^B + \sum_k p_k \rho_k^A \otimes \rho_k^{B C}
$$

并且需要满足下面任意一个不等式：

\begin{align*}
&\left|\rho_{1,8}\right| > \sqrt{\rho_{2,2} \rho_{7,7}} \quad \text{or} \quad \left|\rho_{1,8}\right| > \sqrt{\rho_{3,3} \rho_{6,6}} \quad \text{or} \quad \left|\rho_{1,8}\right| > \sqrt{\rho_{4,4} \rho_{5,5}} \quad \text{or} \quad \\
&\left|\rho_{2,3}\right| > \sqrt{\rho_{1,1} \rho_{4,4}} \quad \text{or} \quad \left|\rho_{2,5}\right| > \sqrt{\rho_{1,1} \rho_{6,6}} \quad \text{or} \quad \left|\rho_{3,5}\right| > \sqrt{\rho_{1,1} \rho_{7,7}}
\end{align*}


In [6]:
import numpy as np


def generate_3_qubit_biseparable_state():
    # 定义子系统的维数
    dim_A = dim_B = dim_C = 2
    dim_AB = dim_AC = dim_BC = 4

    # 生成随机子系统的密度矩阵
    rho_AB = random_density_matrix(dim_AB)
    rho_C = random_density_matrix(dim_C)

    rho_AC = random_density_matrix(dim_AC)
    rho_B = random_density_matrix(dim_B)

    rho_A = random_density_matrix(dim_A)
    rho_BC = random_density_matrix(dim_BC)

    # 定义概率
    p_i = np.random.rand()
    p_j = np.random.rand()
    p_k = np.random.rand()
    total_p = p_i + p_j + p_k
    p_i /= total_p
    p_j /= total_p
    p_k /= total_p

    # 计算双重纠缠态的密度矩阵
    rho = (
        p_i * np.kron(rho_AB, rho_C)
        + p_j * np.kron(rho_AC, rho_B)
        + p_k * np.kron(rho_A, rho_BC)
    )

    # # 归一化 rho 的 trace 为 1
    # rho /= np.trace(rho)

    return rho


# 验证二可分条件
def verify_3_qubit_biseparability_conditions(density_matrix):
    rho_18 = np.abs(density_matrix[0, 7])
    rho_22 = density_matrix[1, 1]
    rho_33 = density_matrix[2, 2]
    rho_44 = density_matrix[3, 3]
    rho_55 = density_matrix[4, 4]
    rho_66 = density_matrix[5, 5]
    rho_77 = density_matrix[6, 6]

    conditions = [
        rho_18 > np.sqrt(rho_22 * rho_77),
        rho_18 > np.sqrt(rho_33 * rho_66),
        rho_18 > np.sqrt(rho_44 * rho_55),
        np.abs(density_matrix[1, 2]) > np.sqrt(density_matrix[0, 0] * rho_44),
        np.abs(density_matrix[1, 4]) > np.sqrt(density_matrix[0, 0] * rho_66),
        np.abs(density_matrix[2, 4]) > np.sqrt(density_matrix[0, 0] * rho_77),
    ]

    return conditions

In [7]:
# 示例: 生成一个 3-qubit 二可分态密度矩阵
three_qubit_biseparable_state = generate_3_qubit_biseparable_state()

# 打印生成的密度矩阵形状
print(
    "Shape of the generated 3-qubit biseparable state density matrix:",
    three_qubit_biseparable_state.shape,
)

# 验证生成的密度矩阵
is_positive_semidefinite, is_trace_one, eigenvalues, trace = verify_density_matrix(
    three_qubit_biseparable_state
)
satisfied_condition = verify_3_qubit_biseparability_conditions(
    three_qubit_biseparable_state
)

# print("\nGenerated 3-qubit biseparable state density matrix:")
# print(three_qubit_biseparable_state)
print("\nVerification Results:")
print("Is positive semidefinite:", is_positive_semidefinite)
print("Is trace one:", is_trace_one)
print("Eigenvalues:", eigenvalues)
print("Trace:", trace)
print("Condition:", satisfied_condition)

Shape of the generated 3-qubit biseparable state density matrix: (8, 8)

Verification Results:
Is positive semidefinite: True
Is trace one: True
Eigenvalues: [ 6.72491709e-01-1.40235860e-17j  2.93052761e-01+1.11261791e-18j
  3.44555296e-02+1.00204780e-18j -4.83921859e-17-6.09246537e-18j
 -1.74289333e-17+1.22918593e-17j -9.75757085e-19+2.19749362e-18j
  3.02268684e-17-7.96717888e-18j  1.87706092e-17-2.11679766e-17j]
Trace: (1-1.5405314748920682e-17j)
Condition: [True, True, False, False, False, True]


In [8]:
# 生成并验证多个3-qubit二可分态密度矩阵
def generate_and_verify_multiple_states(num_matrices):
    satisfied_count = 0
    satisfied_result = []
    not_satisfied_result = []

    for _ in range(num_matrices):
        density_matrix = generate_3_qubit_biseparable_state()
        # _, _, _, _ = verify_density_matrix(density_matrix)
        result = verify_3_qubit_biseparability_conditions(density_matrix)
        if any(result):
            satisfied_count += 1
            satisfied_result.append(result)
        else:
            not_satisfied_result.append(result)

    return satisfied_count, satisfied_result, not_satisfied_result


# 示例: 生成100个包含 5 个随机态的 3-qubit 二可分态密度矩阵
num_matrices = 100
satisfied_count, satisfied_result, not_satisfied_result = (
    generate_and_verify_multiple_states(num_matrices)
)

print(f"\nOut of {num_matrices} generated 3-qubit biseparable state density matrices:")
print(f"Number of matrices that satisfy the conditions: {satisfied_count}")
print(
    f"Number of matrices that do not satisfy the conditions: {num_matrices - satisfied_count}"
)

# if satisfied_result:
#     print("\nConditions that were not satisfied:")
#     for condition in satisfied_result:
#         print(condition)


Out of 100 generated 3-qubit biseparable state density matrices:
Number of matrices that satisfy the conditions: 54
Number of matrices that do not satisfy the conditions: 46


## 纠缠态

生成真正三体纠缠的态：随机生成的量子态如果满足以下不等式则为真正的三体纠缠态

\begin{align*}
&\left|\rho_{2,3}\right|+\left|\rho_{2,5}\right|+\left|\rho_{3,5}\right| > 0.5 \times \left(2 \rho_{1,1}+\rho_{4,4}+\rho_{6,6}+\rho_{7,7}+\rho_{2,2}+\rho_{3,3}+\rho_{5,5}\right) \\
\text{or} \quad &\left|\rho_{1,8}\right| > \sqrt{\rho_{2,2} \rho_{7,7}} + \sqrt{\rho_{3,3} \rho_{6,6}} + \sqrt{\rho_{4,4} \rho_{5,5}}
\end{align*}

In [9]:
def generate_3_qubit_entangled_state():
    # 生成8x8随机密度矩阵
    dim = 8
    rho = random_density_matrix(dim)
    return rho


# 验证纠缠条件
def verify_3_qubit_entangled_conditions(rho):
    left_1 = np.abs(rho[1, 2]) + np.abs(rho[1, 4]) + np.abs(rho[2, 4])
    right_1 = 0.5 * (
        2 * rho[0, 0]
        + rho[3, 3]
        + rho[5, 5]
        + rho[6, 6]
        + rho[1, 1]
        + rho[2, 2]
        + rho[4, 4]
    )
    left_2 = np.abs(rho[0, 7])
    right_2 = (
        np.sqrt(rho[1, 1] * rho[6, 6])
        + np.sqrt(rho[2, 2] * rho[5, 5])
        + np.sqrt(rho[3, 3] * rho[4, 4])
    )

    conditions = [left_1 > right_1, left_2 > right_2]

    return conditions

In [10]:
# 示例: 生成一个 3-qubit 纠缠态密度矩阵
three_qubit_entangled_state = generate_3_qubit_entangled_state()

# 打印生成的密度矩阵形状
print(
    "Shape of the generated 3-qubit entangled state density matrix:",
    three_qubit_entangled_state.shape,
)

# 验证生成的密度矩阵
is_positive_semidefinite, is_trace_one, eigenvalues, trace = verify_density_matrix(
    three_qubit_entangled_state
)
satisfied_condition = verify_3_qubit_entangled_conditions(three_qubit_entangled_state)

# print("\nGenerated 3-qubit entangled state density matrix:")
# print(three_qubit_entangled_state)
print("\nVerification Results:")
print("Is positive semidefinite:", is_positive_semidefinite)
print("Is trace one:", is_trace_one)
print("Eigenvalues:", eigenvalues)
print("Trace:", trace)
print("Condition:", satisfied_condition)

Shape of the generated 3-qubit entangled state density matrix: (8, 8)

Verification Results:
Is positive semidefinite: True
Is trace one: True
Eigenvalues: [-3.50815772e-18+9.16270951e-19j  1.00000000e+00+1.65683582e-16j
 -2.60104237e-17-1.52230602e-17j -1.65128767e-17+1.74947137e-17j
  7.66814520e-18+1.79331234e-17j  5.09795883e-18-5.51640940e-18j
  6.42044734e-18+2.72283953e-18j -2.45204462e-18-1.09104565e-18j]
Trace: (0.9999999999999996+7.691938726130362e-18j)
Condition: [False, False]


In [11]:
# 生成并验证多个3-qubit纠缠态密度矩阵
def generate_and_verify_multiple_states(num_matrices):
    satisfied_count = 0
    satisfied_result = []
    not_satisfied_result = []

    for _ in range(num_matrices):
        density_matrix = generate_3_qubit_entangled_state()
        # _, _, _, _ = verify_density_matrix(density_matrix)
        result = verify_3_qubit_entangled_conditions(density_matrix)
        if any(result):
            satisfied_count += 1
            satisfied_result.append(result)
        else:
            not_satisfied_result.append(result)

    return satisfied_count, satisfied_result, not_satisfied_result


# 示例: 生成100个包含 5 个随机态的 3-qubit 二可分态密度矩阵
num_matrices = 100
satisfied_count, satisfied_result, not_satisfied_result = (
    generate_and_verify_multiple_states(num_matrices)
)

print(f"\nOut of {num_matrices} generated 3-qubit entangled state density matrices:")
print(f"Number of matrices that satisfy the conditions: {satisfied_count}")
print(
    f"Number of matrices that do not satisfy the conditions: {num_matrices - satisfied_count}"
)

# if satisfied_result:
#     print("\nConditions that were not satisfied:")
#     for condition in satisfied_result:
#         print(condition)


Out of 100 generated 3-qubit entangled state density matrices:
Number of matrices that satisfy the conditions: 22
Number of matrices that do not satisfy the conditions: 78


# 数据集生成

## 3-qubit

In [12]:
import random

def store_sub_state_matrix(state_matrix):
    n = state_matrix.shape[0]
    elements = []

    for j in range(n):
        for i in range(j + 1):
            if i == j:
                elements.append(state_matrix[i, j].real)
            else:
                elements.append(state_matrix[i, j].real)
                if state_matrix[i, j].imag != 0:
                    elements.append(complex(0, state_matrix[i, j].imag))

    # 忽略最后一个对角线元素
    elements.pop()

    return elements

def generate_and_verify_multiple_3_qubit_states(num_matrices, state_type):
    satisfied_count = 0
    satisfied_result = []

    if state_type == "entangled":
        generate_state = generate_3_qubit_entangled_state
        verify_conditions = verify_3_qubit_entangled_conditions
    elif state_type == "fully_separable":
        generate_state = generate_3_qubit_fully_separable_state
        verify_conditions = lambda x: [True]  # 对于全可分状态，不需要验证条件
    elif state_type == "biseparable":
        generate_state = generate_3_qubit_biseparable_state
        verify_conditions = verify_3_qubit_biseparability_conditions
    else:
        raise ValueError("Invalid state type")

    while satisfied_count < num_matrices:
        density_matrix = generate_state()
        result = verify_conditions(density_matrix)
        if any(result):
            satisfied_count += 1
            elements = store_sub_state_matrix(density_matrix)
            satisfied_result.append(elements)

    return satisfied_result

def create_3_qubit_dataset(size):
    entangled_states = generate_and_verify_multiple_3_qubit_states(size // 3, "entangled")
    fully_separable_states = generate_and_verify_multiple_3_qubit_states(size // 3, "fully_separable")
    biseparable_states = generate_and_verify_multiple_3_qubit_states(size // 3, "biseparable")

    entangled_labels = [0] * len(entangled_states)
    fully_separable_labels = [1] * len(fully_separable_states)
    biseparable_labels = [2] * len(biseparable_states)

    data_labels = [{"data": state, "label": label} for state, label in zip(entangled_states + fully_separable_states + biseparable_states,
                                                                           entangled_labels + fully_separable_labels + biseparable_labels)]
    
    # 打乱数据集
    random.shuffle(data_labels)
    
    return data_labels

In [13]:
# import numpy as np
# import os

# # 生成30000个3-qubit的数据集，包括每种类型10000个
# size = 30000
# filename = 'dataset/3_qubit_data_labels.npy'

# # 创建目录（如果不存在）
# os.makedirs(os.path.dirname(filename), exist_ok=True)

# data_labels = create_3_qubit_dataset(size)

# # 保存数据和标签到单个文件
# np.save(filename, data_labels)

# print(f"Generated dataset of size {size} with equal distribution of three types of states.")
# print(f"Data and labels have been saved to '{filename}'.")

# # 读取保存的数据并打印第一个数据的形状
# loaded_data_labels = np.load(filename, allow_pickle=True)
# print(f"The number of elements in the first data entry is: {len(loaded_data_labels[0]['data'])}")
# print(f"The label of the first data entry is: {loaded_data_labels[0]['label']}")


In [14]:
import numpy as np
import os


def generate_and_save_3_qubit_dataset(size, filename):
    # 创建目录（如果不存在）
    os.makedirs(os.path.dirname(filename), exist_ok=True)

    # 生成数据集
    data_labels = create_3_qubit_dataset(size)

    # 保存数据和标签到单个文件
    np.save(filename, data_labels)
    print(
        f"Generated dataset of size {size} with equal distribution of three types of states."
    )
    print(f"Data and labels have been saved to '{filename}'.")


# 调用函数生成并保存数据集
generate_and_save_3_qubit_dataset(24000, "dataset/3_qubit_train.npy")
generate_and_save_3_qubit_dataset(6000, "dataset/3_qubit_test.npy")

Generated dataset of size 24000 with equal distribution of three types of states.
Data and labels have been saved to 'dataset/3_qubit_train.npy'.
Generated dataset of size 6000 with equal distribution of three types of states.
Data and labels have been saved to 'dataset/3_qubit_test.npy'.


In [15]:
# 读取保存的数据并打印第一个数据的形状
loaded_data_labels = np.load("dataset/3_qubit_train.npy", allow_pickle=True)
print(f"The number of elements in the first data entry is: {len(loaded_data_labels[0]['data'])}")
print(f"The label of the first data entry is: {loaded_data_labels[0]['label']}")

The number of elements in the first data entry is: 63
The label of the first data entry is: 0
